In [ ]:
import os
import re
from datetime import datetime
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

In [ ]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
if not GroqApiKey:
    raise ValueError("GroqAPI key not found in .env file")

LLM = ChatGroq(
    api_key = GroqApiKey,
    model = "llama-3.3-70b-versatile",
    temperature = 0.7, 
)

In [ ]:
MaxIterations = 5
PassScore = 7

class EssayState(TypedDict):
    Topic: str
    CurrentEssay: str
    CurrentFeedback: str
    CurrentScore: int
    IterationCount: int
    IterationLogs: list   

## Cell 5 — Helper functions

In [ ]:
def makeTimestamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def callLlm(Prompt: str) -> str:
    Response = LLM.invoke(Prompt)
    return Response.content.strip()


def parseScore(CriticResponse: str) -> int:
    
    Match = re.search(r'score[:\s]+([0-9]{1,2})', CriticResponse, re.IGNORECASE)
    if Match:
        return min(int(Match.group(1)), 10)

    Match = re.search(r'([0-9]{1,2})/10', CriticResponse)
    if Match:
        return min(int(Match.group(1)), 10)

    print("  [WARNING] Could not parse score from critic response. Default is 0.")
    return 0

In [ ]:
def writeEssay(State: EssayState) -> dict:
 
    IterationNumber = State["IterationCount"] + 1   

    print(f"\n\n")
    print(f"[{makeTimestamp()}] Iteration {IterationNumber} - write_essay")
    print(f"\n\n")

    if not State["CurrentFeedback"]:
        Prompt = (
            f"Write a well-structured essay on the following topic.\n The essay should have an introduction, at least two body paragraphs, and a conclusion. Aim for approximately 300 words.\n\n Topic: {State['Topic']}"
        )
    else:
        Prompt = (
            f"You previously wrote an essay on the topic below. A critic reviewed it and gave the following feedback. Rewrite the essay to address every point of criticism.\n\n Topic: {State['Topic']}\n\n"
            f"Your previous draft:\n{State['CurrentEssay']}\n\n"
            f"Critic feedback:\n{State['CurrentFeedback']}\n\n"
            f"Now write an improved version of the essay."
        )

    NewEssay = callLlm(Prompt)
    print(f"\n{NewEssay}\n")

    return { "CurrentEssay": NewEssay, "IterationCount": IterationNumber, }

In [ ]:
def critiqueEssay(State: EssayState) -> dict:
   
    print(f"[{makeTimestamp()}] Iteration {State['IterationCount']} - Critique Essay")

    Prompt = (
        f"You are a strict but fair essay critic. Evaluate the essay below on:\n"
        f"1. Clarity and structure\n"
        f"2. Depth of argument\n"
        f"3. Grammar and style\n\n"
        
        f"You must begin your response with exactly this line (fill in N):\n"
        f"Score: N/10\n\n"
        f"Then write 3 to 5 sentences of specific, actionable feedback.\n\n"
        f"Essay:\n{State['CurrentEssay']}"
    )

    CriticResponse = callLlm(Prompt)
    ParsedScore = parseScore(CriticResponse)

    print(f"Score: {ParsedScore}/10")
    print(f"Feedback: {CriticResponse[:120]}")

    LogEntry = (
        f"\n\n"
        
        f"Iteration {State['IterationCount']}  ,  Score: {ParsedScore}/10\n"
        
        f"\n\n"
        
        f"\tEssay\t\n{State['CurrentEssay']}\n\n"
        f"\tCritique\t\n{CriticResponse}\n"
    )

    UpdatedLogs = State["IterationLogs"] + [LogEntry]

    return { "CurrentFeedback": CriticResponse, "CurrentScore": ParsedScore, "IterationLogs": UpdatedLogs,}

In [ ]:
def shouldContinue(State: EssayState) -> str:
    
    Score = State["CurrentScore"]
    Iterations = State["IterationCount"]

    if Score >= PassScore:
        print(f"\n  [ROUTER] Score {Score}/10 >= {PassScore} — essay accepted. Exiting loop.")
        return "end"

    if Iterations >= MaxIterations:
        print(f"\n  [ROUTER] Reached max iterations ({MaxIterations}). Exiting loop.")
        return "end"

    print(f"\n  [ROUTER] Score {Score}/10 < {PassScore} - routing back to Write Essay (iteration {Iterations}/{MaxIterations}).")
    return "write_essay"

In [ ]:
GraphBuilder = StateGraph(EssayState)

GraphBuilder.add_node("write_essay",   writeEssay)
GraphBuilder.add_node("critique_essay", critiqueEssay)

GraphBuilder.add_edge(START, "write_essay")

GraphBuilder.add_edge("write_essay", "critique_essay")

GraphBuilder.add_conditional_edges( "critique_essay", shouldContinue, { "write_essay": "write_essay", "end": END, })

App = GraphBuilder.compile()

In [ ]:
print(App.get_graph().draw_mermaid())

In [ ]:
EssayTopic = input("Enter the essay topic: ").strip()

if not EssayTopic:
    raise ValueError("Topic cannot be empty.")

InitialState = {
    "Topic": EssayTopic,
    "CurrentEssay": "",
    "CurrentFeedback": "",
    "CurrentScore": 0,
    "IterationCount": 0,
    "IterationLogs": [],
}

print(f"\n[{makeTimestamp()}] Starting self-correcting essay workflow...")
print(f"Topic: '{EssayTopic}'")
print(f"Pass score: {PassScore}/10 | Max iterations: {MaxIterations}\n")

FinalState = App.invoke(InitialState)

print(f"\n[{makeTimestamp()}] Workflow complete.")
print(f"Final score: {FinalState['CurrentScore']}/10 after {FinalState['IterationCount']} iteration.")

In [ ]:
def saveReport(State: EssayState):
   
    Timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    OutputFileName = f"essay_report_{Timestamp}.txt"

    if State["CurrentScore"] >= PassScore:
        ExitReason = f"Score {State['CurrentScore']}/10 met the pass threshold of {PassScore}/10"
    else:
        ExitReason = f"Reached maximum of {MaxIterations} iterations (final score: {State['CurrentScore']}/10)"

    FinalEssaySection = (
        f"\n\n"
        
        f"Self Correcting Essay Writer - Final Report\n"
        
        f"Generated: {makeTimestamp()}\n"
        f"Topic: {State['Topic']}\n"
        f"Iterations: {State['IterationCount']}\n"
        f"Exit: {ExitReason}\n"
        
        f"\n\n"
        
        f"Final Accepted Essay\n"
        
        f"\n\n"
        
        f"{State['CurrentEssay']}\n\n"
    )

    # Iteration history section
    HistorySection = (
        f"\n\n"
        f"Full iteration log\n"
        f"\n\n"
    )

    for LogBlock in State["IterationLogs"]:
        HistorySection += LogBlock + "\n"

    with open(OutputFileName, "w", encoding="utf-8") as OutputFile:
        OutputFile.write(FinalEssaySection)
        OutputFile.write(HistorySection)

    print(f"Report saved to: {OutputFileName}")
    return OutputFileName

SavedFile = saveReport(FinalState)